# Compound AI Systems Exercise — Assessment 05

**Role context:** Researcher interview, live notebook prototyping  
**Time limit: 45 minutes. No AI coding assistance.**


## Problem Statement

Employees file **ambiguous, multi-intent IT tickets** over Slack. A junior automation bot must: unpack intents, prioritize by urgency, ask clarification only when necessary, and propose a **tool-call plan** plus a user-facing reply.

Build a compound system for ambiguous request resolution — not a single classification label.


## Functional Requirements

**Input**
- `ticket: dict` — ticket_id, requester_id, subject, body, channel, created_at
- `employee_directory: list[dict]` — user metadata
- `tool_catalog: str` — available tools (propose calls; do not execute)

**Output**
```python
{
  "intents": [
    {
      "name": "vpn_access" | "mfa_issue" | "software_install" | "other",
      "priority": "p1" | "p2" | "p3",
      "urgency_reason": "...",
      "status": "actionable" | "needs_clarification" | "deferred"
    }
  ],
  "clarifying_questions": ["only if blocking"],
  "proposed_tool_calls": [
    {"tool": "reset_vpn_profile", "args": {"user_id": "u_19"}, "depends_on": []}
  ],
  "execution_order": ["reset_vpn_profile", "..."],
  "user_reply": "message to send now",
  "internal_notes": "for human IT agents",
  "assumptions": ["..."]
}
```

**Constraints**
- Prefer unblocking access issues before convenience installs when the user signals priority.
- Do not invent tools outside the catalog.
- Resolve `requester_id` via directory for asset/manager fields when needed.
- Ask clarifying questions only for **blocking** unknowns (e.g., exact package name / deadline if material).
- If the user said to ignore the plugin when VPN is on fire, reflect that in deferral.

**Edge cases**
- Multiple intents with explicit priority hint.
- Ambiguous deadline ("tomorrow" vs "Thursday").
- Soft MFA mention without clear failure mode.

**Desired behavior**
Safe automation proposal a human could approve in one glance.


## Available Tools & Starter Code


In [ ]:
import json
import os
import re
from typing import Any, Optional

# ---------------------------------------------------------------------------
# LLM configuration
# ---------------------------------------------------------------------------
# Set one of the following before running (or export in your shell):
#
#   export OPENAI_API_KEY="sk-..."
#   # optional: export OPENAI_MODEL="gpt-4o-mini"
#
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   # optional: export ANTHROPIC_MODEL="claude-3-5-haiku-latest"
#
# If neither key is set, call_llm runs in MOCK mode so the notebook still
# executes for smoke-testing. Mock responses are placeholders — they are not
# a substitute for a real model during practice.
# ---------------------------------------------------------------------------

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")

USE_MOCK = not (OPENAI_API_KEY or ANTHROPIC_API_KEY)
if USE_MOCK:
    print("⚠️  No API key found. call_llm is in MOCK mode.")
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"✓ Using {provider} API")


def _mock_response(prompt: str, system_prompt: str = "") -> str:
    """Lightweight deterministic mock for offline smoke tests."""
    combined = f"{system_prompt}\n{prompt}".lower()
    # Return minimal JSON-ish or text so downstream parsing can be exercised.
    if "json" in combined:
        return json.dumps({"mock": True, "note": "Replace with real API key for useful output", "items": []})
    return "[MOCK RESPONSE] Configure OPENAI_API_KEY or ANTHROPIC_API_KEY for real model output."


def call_llm(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.2,
    max_tokens: int = 2000,
) -> str:
    """Call a chat LLM. Credentials come from environment variables only."""
    if USE_MOCK:
        return _mock_response(prompt, system_prompt)

    try:
        if OPENAI_API_KEY:
            from openai import OpenAI

            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        from anthropic import Anthropic

        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        ).strip()
    except Exception as e:
        return f"[API ERROR] {type(e).__name__}: {e}"


def extract_json(text: str) -> Any:
    """Best-effort parse of a JSON object/array from model output."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except json.JSONDecodeError:
            pass
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Could not parse JSON from model output:\n{text[:500]}")


# Smoke test
print("call_llm ready.")
print(call_llm("Reply with exactly: OK", system_prompt="Be terse.")[:80])


In [ ]:
EMPLOYEE_DIRECTORY = [
  {
    "user_id": "u_88",
    "name": "Sam Ortiz",
    "dept": "Finance",
    "manager": "u_12",
    "laptop_asset": "L-2031"
  },
  {
    "user_id": "u_19",
    "name": "Alex Kim",
    "dept": "Engineering",
    "manager": "u_02",
    "laptop_asset": "L-7781"
  }
]

TICKET = {
  "ticket_id": "IT-55401",
  "requester_id": "u_19",
  "subject": "urgent cant access + need software",
  "body": "hey my vpn keeps failing after the password reset yesterday and also can someone install the design plugin on my machine before the client demo tomorrow? or wait maybe the demo is Thursday. also I think my MFA soft token is weird. If vpn is a bigger fire ignore the plugin. thanks",
  "channel": "slack_dm",
  "created_at": "2026-10-13T16:05:00Z"
}

TOOL_CATALOG = """Available resolution tools (you cannot execute them; propose calls only):
1. reset_vpn_profile(user_id)
2. push_mfa_reenrollment(user_id)
3. install_software(asset_id, package_name)
4. schedule_technician(asset_id, window)
5. request_manager_approval(manager_id, reason)
6. reply_to_user(ticket_id, message)"""

print(TICKET["subject"])


## Coding Section


In [ ]:
def resolve_ambiguous_ticket(ticket: dict, employee_directory: list, tool_catalog: str) -> dict:
    """Resolve an ambiguous multi-intent IT ticket into a plan and reply.

    Returns:
        dict matching Functional Requirements.
    """
    # TODO: Implement your compound AI system here.
    raise NotImplementedError("Implement resolve_ambiguous_ticket")


## Example Input / Output

For `TICKET`, a strong solution typically:
- Identifies VPN (+ maybe MFA) as higher priority than software install
- Defers or clarifies the design plugin (package name / deadline)
- Proposes `reset_vpn_profile` / MFA tools before install
- Writes a calm Slack reply acknowledging the priority split

Exact tool choices may vary; inventing tools is not allowed.


## Run Your Solution


In [ ]:
# result = resolve_ambiguous_ticket(TICKET, EMPLOYEE_DIRECTORY, TOOL_CATALOG)
# print(json.dumps(result, indent=2))


## Time Limit

**Time limit: 45 minutes. No AI coding assistance.**
